# Blocks 1–5 — Colab pipeline (live tree)

Opening a notebook from GitHub **does not** include `src/`. **Runtime → Run all.** The first code cell must print `BOOTSTRAP_V4` and `med_doc: .../src/med_doc/__init__.py`. If you still see `No module named med_doc`, you are on an old notebook copy: File → Open notebook from GitHub → `RwaRwa599/epq3` branch **`block1`** → `Run_in_Colab.ipynb`, then **Runtime → Disconnect and delete runtime**.

Older `block1/`, `block2/`, `block3/` folders are snapshots. Prefer this notebook + `notebooks/Block_*.ipynb`.

**Do not upload clinic PHI to Colab.**


Per-block notebooks: [1](https://colab.research.google.com/github/RwaRwa599/epq3/blob/block1/notebooks/Block_1_Document_Normalization.ipynb) · [2](https://colab.research.google.com/github/RwaRwa599/epq3/blob/block1/notebooks/Block_2_Knowledge_Graph.ipynb) · [3](https://colab.research.google.com/github/RwaRwa599/epq3/blob/block1/notebooks/Block_3_Marks_and_HTR.ipynb) · [4](https://colab.research.google.com/github/RwaRwa599/epq3/blob/block1/notebooks/Block_4_KG_Rescoring.ipynb) · [5](https://colab.research.google.com/github/RwaRwa599/epq3/blob/block1/notebooks/Block_5_Review_and_LIS.ipynb)


## 0. Download src/med_doc (zipball)


In [ ]:
# BOOTSTRAP_V4 — zipball (no git, no %pip -e; editable install restarts Colab mid-cell)
import os
import shutil
import sys
import urllib.request
import zipfile
from pathlib import Path

CONTENT = Path("/content") if Path("/content").is_dir() else Path.cwd()
REPO = CONTENT / "epq3"
SRC = REPO / "src"
URL = "https://codeload.github.com/RwaRwa599/epq3/zip/refs/heads/block1"

if not (SRC / "med_doc" / "__init__.py").is_file():
    zpath = CONTENT / "epq3-block1.zip"
    print("Downloading", URL)
    urllib.request.urlretrieve(URL, zpath)
    extract = CONTENT / "_epq3_extract"
    if extract.exists():
        shutil.rmtree(extract)
    extract.mkdir()
    with zipfile.ZipFile(zpath) as zf:
        zf.extractall(extract)
    found = list(extract.glob("*/src/med_doc/__init__.py"))
    if not found:
        raise RuntimeError(f"zip missing src/med_doc: {list(extract.iterdir())}")
    unpacked = found[0].parents[2]
    if REPO.exists():
        shutil.rmtree(REPO)
    shutil.move(str(unpacked), str(REPO))
    shutil.rmtree(extract, ignore_errors=True)
    zpath.unlink(missing_ok=True)

sys.path.insert(0, str(SRC.resolve()))
os.chdir(REPO)
import med_doc

print("BOOTSTRAP_V4")
print("cwd:", os.getcwd())
print("med_doc:", med_doc.__file__)


In [ ]:
# Runtime deps via pip CLI (not %pip / not pip -e — those restart Colab mid-run).
import subprocess
import sys

subprocess.check_call(
    [sys.executable, "-m", "pip", "install", "-q", "opencv-python-headless", "pydantic", "matplotlib", "Pillow", "numpy"]
)


The repo is public. No GitHub token is required. Opening this notebook from GitHub still does not include `src/` — the first cell downloads the zipball.


In [ ]:
import os
import sys
from pathlib import Path

def _guard():
    content = Path("/content") if Path("/content").is_dir() else Path.cwd()
    hits = list(content.glob("epq3/src/med_doc/__init__.py"))
    hits += list(content.glob("*/src/med_doc/__init__.py"))
    if not hits:
        raise ModuleNotFoundError(
            "med_doc missing. Run the FIRST code cell until it prints BOOTSTRAP_V4 "
            "and med_doc: .../src/med_doc/__init__.py. Open Run_in_Colab.ipynb from "
            "GitHub branch block1. Runtime → Disconnect and delete runtime, then Run all."
        )
    src = hits[0].parents[1]
    root = hits[0].parents[2]
    os.chdir(root)
    sp = str(src.resolve())
    if sp not in sys.path:
        sys.path.insert(0, sp)
    return sp
_guard()

from pathlib import Path
import json
import cv2
import matplotlib.pyplot as plt

try:
    from google.colab import files as colab_files
except Exception:
    colab_files = None

OUT = Path("/content/pipeline") if Path("/content").is_dir() else Path("outputs/colab_pipeline")
OUT.mkdir(parents=True, exist_ok=True)

def show_rgb(path, title="", figsize=(10, 8)):
    path = Path(path)
    if not path.exists():
        print("missing", path)
        return
    bgr = cv2.imread(str(path))
    if bgr is None:
        print("unreadable", path)
        return
    rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
    plt.figure(figsize=figsize)
    plt.imshow(rgb)
    plt.title(title or path.name)
    plt.axis("off")
    plt.show()

def download(path):
    path = Path(path)
    print(path, f"({path.stat().st_size / 1024:.1f} KB)" if path.exists() else "missing")
    if colab_files and path.exists():
        colab_files.download(str(path))

def demo_sheet() -> Path:
    from med_doc.paths import SYNTHETIC_DIR
    sheet = SYNTHETIC_DIR / "lab_request_v0_blank.png"
    assert sheet.exists(), sheet
    return sheet


## 1. Choose a batch of images

Same input shapes as Block 1a: a **folder**, a **ZIP of photos**, a **list of paths**, or Colab multi-upload.

Do **not** upload clinic PHI. Default: two copies of the synthetic blank.


In [ ]:
import os
import sys
from pathlib import Path

def _guard():
    content = Path("/content") if Path("/content").is_dir() else Path.cwd()
    hits = list(content.glob("epq3/src/med_doc/__init__.py"))
    hits += list(content.glob("*/src/med_doc/__init__.py"))
    if not hits:
        raise ModuleNotFoundError(
            "med_doc missing. Run the FIRST code cell until it prints BOOTSTRAP_V4 "
            "and med_doc: .../src/med_doc/__init__.py. Open Run_in_Colab.ipynb from "
            "GitHub branch block1. Runtime → Disconnect and delete runtime, then Run all."
        )
    src = hits[0].parents[1]
    root = hits[0].parents[2]
    os.chdir(root)
    sp = str(src.resolve())
    if sp not in sys.path:
        sys.path.insert(0, sp)
    return sp
_guard()

import shutil
from med_doc.pipeline import run_blocks_1_to_5

USE_UPLOAD = False

OUTPUT_MODE = "user"  # "user" = one order.json; "dev" = block1/3/4/5 zips
BATCH_DIR = None  # e.g. Path("/content/photos") or Path("photos.zip")

if USE_UPLOAD:
    from google.colab import files
    uploaded = files.upload()
    names = list(uploaded.keys())
    if len(names) == 1 and names[0].lower().endswith(".zip"):
        batch_input = names[0]
    else:
        batch_input = names
elif BATCH_DIR is not None:
    batch_input = BATCH_DIR
else:
    sheet = demo_sheet()
    raw = OUT / "raw_batch"
    raw.mkdir(parents=True, exist_ok=True)
    shutil.copy2(sheet, raw / "form_a.png")
    shutil.copy2(sheet, raw / "form_b.png")
    batch_input = raw

print("batch_input:", batch_input)


## 2. Run Blocks 1–5 on the whole batch

`run_blocks_1_to_5` = `normalize_batch` → Block 3 drafts → Block 4 KG → Block 5 orders.


In [ ]:
import os
import sys
from pathlib import Path

def _guard():
    content = Path("/content") if Path("/content").is_dir() else Path.cwd()
    hits = list(content.glob("epq3/src/med_doc/__init__.py"))
    hits += list(content.glob("*/src/med_doc/__init__.py"))
    if not hits:
        raise ModuleNotFoundError(
            "med_doc missing. Run the FIRST code cell until it prints BOOTSTRAP_V4 "
            "and med_doc: .../src/med_doc/__init__.py. Open Run_in_Colab.ipynb from "
            "GitHub branch block1. Runtime → Disconnect and delete runtime, then Run all."
        )
    src = hits[0].parents[1]
    root = hits[0].parents[2]
    os.chdir(root)
    sp = str(src.resolve())
    if sp not in sys.path:
        sys.path.insert(0, sp)
    return sp
_guard()

from med_doc.pipeline import run_blocks_1_to_5

pipe = run_blocks_1_to_5(
    batch_input,
    output_dir=OUT,
    backend="lexicon",
    patch_missing_edta=True,  # demo nurse patch when EDTA crop is empty
    output_mode=OUTPUT_MODE,
)
print("mode", pipe["output_mode"], "docs", pipe["block5"]["manifest"]["total_documents"])
print("output_json", pipe.get("output_json"))
for row in pipe["block5"]["manifest"]["documents"]:
    print(row["doc_id"], "ticked", row.get("n_ticked"), "needs_review", row.get("needs_review"),
          "observed", row.get("observed_tubes"))


## 3. Inspect first document


In [ ]:
import os
import sys
from pathlib import Path

def _guard():
    content = Path("/content") if Path("/content").is_dir() else Path.cwd()
    hits = list(content.glob("epq3/src/med_doc/__init__.py"))
    hits += list(content.glob("*/src/med_doc/__init__.py"))
    if not hits:
        raise ModuleNotFoundError(
            "med_doc missing. Run the FIRST code cell until it prints BOOTSTRAP_V4 "
            "and med_doc: .../src/med_doc/__init__.py. Open Run_in_Colab.ipynb from "
            "GitHub branch block1. Runtime → Disconnect and delete runtime, then Run all."
        )
    src = hits[0].parents[1]
    root = hits[0].parents[2]
    os.chdir(root)
    sp = str(src.resolve())
    if sp not in sys.path:
        sys.path.insert(0, sp)
    return sp
_guard()


import json

bundle = json.loads(Path(pipe["output_json"]).read_text())
print("orders", bundle["total_documents"], "file", pipe["output_json"])
for order in bundle["orders"]:
    print(order["doc_id"], "needs_review", order["needs_review"], "observed", order["observed_tubes"], "tests", order["ordered_tests"])

if pipe["output_mode"] == "dev":
    docs = pipe["block5"]["manifest"]["documents"]
    doc_id = docs[0]["doc_id"]
    show_rgb(OUT / "b1" / "docs" / doc_id / "overlay.png", f"Block 1 overlay — {doc_id}", figsize=(12, 10))
    hyp = json.loads((OUT / "b3" / "docs" / doc_id / "hypotheses.json").read_text())
    pred = json.loads((OUT / "b4" / "docs" / doc_id / "prediction.json").read_text())
    print("marked", [k for k, v in hyp["nonverbal"].items() if v["is_marked"]])
    print("tube source", hyp["verbal"].get("tube_edta", {}).get("source"))
    print("B4 observed", pred.get("observed_tubes"), "expected", pred.get("expected_tubes"))
    assert hyp["verbal"].get("tube_edta", {}).get("source") != "prior_expected"


## 4. Download — `order.json` (user) or block ZIPs (dev)


In [ ]:
import os
import sys
from pathlib import Path

def _guard():
    content = Path("/content") if Path("/content").is_dir() else Path.cwd()
    hits = list(content.glob("epq3/src/med_doc/__init__.py"))
    hits += list(content.glob("*/src/med_doc/__init__.py"))
    if not hits:
        raise ModuleNotFoundError(
            "med_doc missing. Run the FIRST code cell until it prints BOOTSTRAP_V4 "
            "and med_doc: .../src/med_doc/__init__.py. Open Run_in_Colab.ipynb from "
            "GitHub branch block1. Runtime → Disconnect and delete runtime, then Run all."
        )
    src = hits[0].parents[1]
    root = hits[0].parents[2]
    os.chdir(root)
    sp = str(src.resolve())
    if sp not in sys.path:
        sys.path.insert(0, sp)
    return sp
_guard()


if pipe["output_mode"] == "dev":
    for name in ("block1.zip", "block3.zip", "block4.zip", "block5.zip"):
        download(OUT / name)
else:
    download(OUT / "order.json")
